# Deduplicate Secondary Timeseries Configuration
This notebook will drop the duplicate values for a single configuration in the seocnrday timeseries.
It does so by querying out the configuration, depulicating, deleting the original configuration, and 
insert the dedeuped data back.

In [1]:
import os
from pathlib import Path
import tempfile
import shutil

import teehr
from teehr.models.filters import TableFilter

from pyspark.sql import functions as F
from pyspark.sql.window import Window

teehr.__version__

'0.6.1'

In [2]:
from teehr.evaluation.spark_session_utils import create_spark_session
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=64,
    executor_memory="32g",
    executor_cores=4,
    aws_profile="default",
    update_configs={
        "spark.sql.shuffle.partitions": 512,
    }
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.5.133
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 64
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 32g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 4
INFO:teehr.evaluati

In [3]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


In [4]:
config_name = "nrds_v22_cfenom_medium_range"

In [5]:
filters = [
    TableFilter(
        column='configuration_name',
        operator='=',
        value=config_name
    )
]

In [6]:
source = ev.secondary_timeseries.filter(filters).to_sdf()    

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter [TableFilter(column='configuration_name', operator=<FilterOperators.eq: '='>, value='nrds_v22_cfenom_medium_range')].


In [7]:
key_cols = ev.secondary_timeseries.uniqueness_fields
key_cols

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.


['location_id',
 'value_time',
 'reference_time',
 'variable_name',
 'unit_name',
 'configuration_name',
 'member']

In [8]:
duplicate_groups = (
    source.groupBy(key_cols)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_group_count = duplicate_groups.count()

duplicate_row_count = (
    duplicate_groups
    .select(F.sum(F.col("count") - 1).alias("duplicate_rows"))
    .collect()[0]["duplicate_rows"]
)

print(f"duplicate groups: {duplicate_group_count}")
print(f"duplicate rows beyond first occurrence: {duplicate_row_count}")

duplicate groups: 13513680
duplicate rows beyond first occurrence: 13513680


In [9]:
deduped = source.drop_duplicates(key_cols)

before_count = source.count()
after_count = deduped.count()

print(f"rows before: {before_count}")
print(f"rows after: {after_count}")
print(f"duplicates removed: {before_count - after_count}")

rows before: 826779672
rows after: 813265992
duplicates removed: 13513680


In [10]:
# Optional preview of rows that will be deleted
rows_to_delete = ev.secondary_timeseries.delete(
    filters=filters,
    dry_run=True,
)
print(f"rows to delete: {rows_to_delete.count()}")

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.


rows to delete: 826779672


In [11]:
# Delete the bad slice
deleted_count = ev.secondary_timeseries.delete(filters=filters)
print(f"deleted rows: {deleted_count}")

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.


deleted rows: 826779672


In [12]:
# Insert the cleaned slice back
ev.secondary_timeseries.load_dataframe(
    deduped,
    write_mode="append",
    drop_duplicates=False,
)

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.validate:Start enforcing dataframe schema.
INFO:teehr.evaluation.validate:Validating DataFrame against schema.
INFO:teehr.evaluation.validate:Enforcing foreign key constraints.
INFO:teehr.evaluation.validate:Finished enforcing dataframe schema in 72.960 seconds.
INFO:teehr.evaluation.write:Start writing to warehouse table 'secondary_timeseries'.
INFO:teehr.e

In [13]:
spark.stop()